## Named volumes — the default choice

A **named volume** is storage managed entirely by Docker: you give it a name, Docker decides where on disk it lives, and you mount it into containers by name. This is the default you reach for.

```bash
docker volume create pgdata
docker volume ls
docker volume inspect pgdata      # shows mountpoint, driver (local), labels
docker volume rm pgdata
```

Mount it into a container with `-v <name>:<path>`:

```bash
docker run -d --name db \
  -v pgdata:/var/lib/postgresql/data \
  -e POSTGRES_PASSWORD=secret postgres:16
```

That reads "mount the named volume `pgdata` at `/var/lib/postgresql/data`." Docker **auto-creates** `pgdata` if it doesn't exist — convenient, though production scripts do the explicit `docker volume create`. Crucially, `docker rm db` later **doesn't touch `pgdata`**: start a fresh Postgres with the same mount and your data is right there. That's persistence.

**One gotcha — mounting over a non-empty path.** If the container path already had files in the image and you mount a **fresh, empty** named volume there, Docker *copies the image's files into the volume* on first use. (This is how official DB images seed an initial data dir.) A bind mount does **not** do this — it just masks whatever was there.

**Two ways to get a volume without naming it — both can surprise you:**

- **Anonymous volumes** — `-v /container/path` with no source name creates a volume with a random hex name. These pile up forever unless you use `--rm` (which drops them with the container) or `docker volume prune`.
- **The `VOLUME` instruction** in a Dockerfile (`VOLUME /var/lib/postgresql/data`) declares a path *must* be a volume; run the image without a mount there and Docker auto-creates an anonymous one. It's how `postgres`, `mysql`, and `mongo` stop you from putting a database on the throwaway writable layer — helpful, but a steady source of stray anonymous volumes.